# 01 - AI Business Intelligence: Dataset Exploration

এই notebook-এ Store Sales এবং Online Retail dataset লোড করে data quality, schema, missing values, duplicate rows, date coverage এবং basic preprocessing পরীক্ষা করা হবে। এখানে কোনো AI provider বা database connection দরকার নেই। বড় CSV-র ক্ষেত্রে inspection sample ব্যবহার হবে, তবে পুরো CSV-র row count memory-efficient ভাবে গণনা করা হবে।

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATA_FILES = [
    'train.csv', 'test.csv', 'stores.csv', 'transactions.csv',
    'holidays_events.csv', 'oil.csv', 'sample_submission.csv',
    'Online Retail.xlsx',
]

def find_data_dir():
    roots = [Path.cwd(), *Path.cwd().parents]
    candidates = []
    for root in roots:
        candidates.extend([root / 'dataset', root / 'notebooks' / 'dataset', root / 'ai-service' / 'notebooks'])
    for candidate in candidates:
        if any((candidate / name).is_file() for name in DATA_FILES):
            return candidate
    raise FileNotFoundError(
        'Dataset folder not found. Expected files in notebooks/dataset. '
        'Move or copy the datasets there, then rerun this cell.'
    )

DATA_DIR = find_data_dir()
print(f'Dataset folder: {DATA_DIR}')

Dataset folder: e:\9th semi\AI Project\ai-service\notebooks\dataset


## File inventory and dataset profile

`train.csv`, `test.csv`, and `sample_submission.csv` are capped to a preview sample for profiling. The full row count is counted by iterating over each CSV, not by loading the whole file.

In [2]:
LARGE_CSVS = {'train.csv', 'test.csv', 'sample_submission.csv'}
PREVIEW_ROWS = 100_000
datasets = {}
inventory = []

def count_csv_rows(path):
    with path.open('r', encoding='utf-8-sig', errors='replace') as handle:
        return max(sum(1 for _ in handle) - 1, 0)

for name in DATA_FILES:
    path = DATA_DIR / name
    if not path.is_file():
        print(f'MISSING: {name}')
        continue
    size_mb = path.stat().st_size / (1024 ** 2)
    if path.suffix.lower() == '.csv':
        full_rows = count_csv_rows(path)
        row_limit = PREVIEW_ROWS if name in LARGE_CSVS else None
        frame = pd.read_csv(path, nrows=row_limit, low_memory=False)
        sampled = row_limit is not None and full_rows > len(frame)
    else:
        frame = pd.read_excel(path, nrows=PREVIEW_ROWS)
        full_rows = None
        sampled = len(frame) == PREVIEW_ROWS
    datasets[name] = frame
    inventory.append({
        'file': name,
        'file_size_mb': round(size_mb, 2),
        'full_csv_rows': full_rows,
        'loaded_rows': len(frame),
        'sampled': sampled,
        'columns': len(frame.columns),
    })

display(pd.DataFrame(inventory))

,file,file_size_mb,full_csv_rows,loaded_rows,sampled,columns
0,train.csv,116.16,3000888.0,100000,True,6
1,test.csv,0.97,28512.0,28512,False,5
2,stores.csv,0.00,54.0,54,False,5
3,transactions.csv,1.48,83488.0,83488,False,3
4,holidays_events.csv,0.02,350.0,350,False,6
5,oil.csv,0.02,1218.0,1218,False,2
6,sample_submission.csv,0.33,28512.0,28512,False,2
7,Online Retail.xlsx,22.62,NaN,100000,True,8


In [3]:
def profile_frame(name, frame):
    print(f'\n### {name}')
    print(f'Loaded shape: {frame.shape}')
    print(f'Duplicate rows in loaded data: {frame.duplicated().sum():,}')
    profile = pd.DataFrame({
        'dtype': frame.dtypes.astype(str),
        'missing': frame.isna().sum(),
        'missing_pct': (frame.isna().mean() * 100).round(2),
        'unique_values': frame.nunique(dropna=True),
    })
    display(profile)
    date_columns = [
        column for column in frame.columns
        if any(token in column.lower() for token in ('date', 'time'))
    ]
    for column in date_columns:
        parsed = pd.to_datetime(frame[column], errors='coerce')
        if parsed.notna().any():
            print(f'{column} range: {parsed.min()} to {parsed.max()} '
                  f'({parsed.isna().sum():,} unparsed/missing in loaded data)')
    display(frame.head(5))

for name, frame in datasets.items():
    profile_frame(name, frame)


### train.csv
Loaded shape: (100000, 6)
Duplicate rows in loaded data: 0


,dtype,missing,missing_pct,unique_values
id,int64,0,0.0,100000
date,str,0,0.0,57
store_nbr,int64,0,0.0,54
family,str,0,0.0,33
sales,float64,0,0.0,15007
onpromotion,int64,0,0.0,1


date range: 2013-01-01 00:00:00 to 2013-02-26 00:00:00 (0 unparsed/missing in loaded data)


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0



### test.csv
Loaded shape: (28512, 5)
Duplicate rows in loaded data: 0


,dtype,missing,missing_pct,unique_values
id,int64,0,0.0,28512
date,str,0,0.0,16
store_nbr,int64,0,0.0,54
family,str,0,0.0,33
onpromotion,int64,0,0.0,212


... (20 outputs left out)


### Online Retail.xlsx
Loaded shape: (100000, 8)
Duplicate rows in loaded data: 924


,dtype,missing,missing_pct,unique_values
InvoiceNo,object,0,0.00,4657
StockCode,object,0,0.00,3143
Description,str,299,0.30,3049
Quantity,int64,0,0.00,299
InvoiceDate,datetime64[us],0,0.00,4207
UnitPrice,float64,0,0.00,572
CustomerID,float64,34915,34.92,1664
Country,str,0,0.00,29


InvoiceDate range: 2010-12-01 08:26:00 to 2011-02-23 13:37:00 (0 unparsed/missing in loaded data)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Cleaning and preprocessing

These steps build cleaned copies from the loaded data (large files use the preview sample configured above). Exact duplicate rows and invalid dates are counted. Retail returns/cancellations are kept separately from positive sales; missing customer IDs remain unknown, and missing product descriptions are labeled `Unknown product`.


In [ ]:
cleaned_datasets = {}
cleaning_records = []

if 'train.csv' in datasets:
    store_source = datasets['train.csv'].copy()
    store_input_rows = len(store_source)
    store_duplicates = int(store_source.duplicated().sum())
    store_sales_cleaned = store_source.drop_duplicates().copy()

    invalid_store_dates = 0
    if 'date' in store_sales_cleaned:
        store_sales_cleaned['date'] = pd.to_datetime(
            store_sales_cleaned['date'], errors='coerce'
        )
        invalid_store_dates = int(store_sales_cleaned['date'].isna().sum())
        store_sales_cleaned = store_sales_cleaned.dropna(subset=['date']).copy()

    if 'family' in store_sales_cleaned:
        store_sales_cleaned['family'] = (
            store_sales_cleaned['family'].astype('string').str.strip()
            .replace('', pd.NA).fillna('Unknown family')
        )
    for column in ('sales', 'onpromotion'):
        if column in store_sales_cleaned:
            store_sales_cleaned[column] = pd.to_numeric(
                store_sales_cleaned[column], errors='coerce'
            )

    cleaned_datasets['train.csv'] = store_sales_cleaned
    cleaning_records.append({
        'dataset': 'Store Sales (loaded preview)',
        'input_rows': store_input_rows,
        'exact_duplicates_removed': store_duplicates,
        'invalid_dates_removed': invalid_store_dates,
        'invalid_numeric_rows_removed': 0,
        'nonpositive_prices_removed': 0,
        'returns_or_cancellations_separated': 0,
        'zero_quantity_rows_excluded': 0,
        'retained_sales_rows': len(store_sales_cleaned),
    })

    print('Store Sales cleaned preview:', store_sales_cleaned.shape)
    print('Store Sales retained missing values by column:')
    display(store_sales_cleaned.isna().sum().rename('missing').to_frame())
    if {'family', 'sales'}.issubset(store_sales_cleaned.columns):
        print('Top product families by sales:')
        family_summary = (
            store_sales_cleaned.groupby('family', dropna=False)
            .agg(rows=('sales', 'size'), total_sales=('sales', 'sum'),
                 average_sales=('sales', 'mean'))
            .sort_values('total_sales', ascending=False)
        )
        display(family_summary.head(10))
    if {'store_nbr', 'sales'}.issubset(store_sales_cleaned.columns):
        print('Top stores by sales:')
        store_summary = (
            store_sales_cleaned.groupby('store_nbr', dropna=False)
            .agg(rows=('sales', 'size'), total_sales=('sales', 'sum'))
            .sort_values('total_sales', ascending=False)
        )
        display(store_summary.head(10))
    if {'onpromotion', 'sales'}.issubset(store_sales_cleaned.columns):
        promotion_status = pd.Series('Unknown', index=store_sales_cleaned.index)
        promotion_status.loc[store_sales_cleaned['onpromotion'].gt(0)] = 'On promotion'
        promotion_status.loc[store_sales_cleaned['onpromotion'].eq(0)] = 'No promotion'
        promotion_summary = (
            store_sales_cleaned.assign(promotion_status=promotion_status)
            .groupby('promotion_status', dropna=False)
            .agg(rows=('sales', 'size'), total_sales=('sales', 'sum'),
                 average_sales=('sales', 'mean'))
        )
        print('Sales by promotion status:')
        display(promotion_summary)

if 'Online Retail.xlsx' in datasets:
    retail_source = datasets['Online Retail.xlsx'].copy()
    retail_input_rows = len(retail_source)
    retail_duplicates = int(retail_source.duplicated().sum())
    retail = retail_source.drop_duplicates().copy()
    retail_missing = retail.isna().sum()
    missing_customer_ids = int(retail_missing.get('CustomerID', 0))
    if 'Description' in retail:
        descriptions = retail['Description'].astype('string').str.strip()
        missing_descriptions = int((retail['Description'].isna() | descriptions.eq('').fillna(False)).sum())
    else:
        missing_descriptions = 0
    required_retail_columns = {'Quantity', 'UnitPrice', 'InvoiceDate'}
    missing_retail_columns = required_retail_columns - set(retail.columns)

    print('Online Retail loaded preview:', retail_source.shape)
    print('Missing values before cleaning:')
    display(retail_missing.rename('missing').to_frame())

    if missing_retail_columns:
        print('Online Retail cleaning skipped; required columns missing:',
              ', '.join(sorted(missing_retail_columns)))
    else:
        retail['InvoiceDate'] = pd.to_datetime(
            retail['InvoiceDate'], errors='coerce'
        )
        invalid_retail_dates = int(retail['InvoiceDate'].isna().sum())
        retail = retail.dropna(subset=['InvoiceDate']).copy()

        retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
        retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
        invalid_retail_numeric = int(
            (retail['Quantity'].isna() | retail['UnitPrice'].isna()).sum()
        )
        retail = retail.dropna(subset=['Quantity', 'UnitPrice']).copy()
        nonpositive_retail_prices = int(retail['UnitPrice'].le(0).sum())
        retail = retail.loc[retail['UnitPrice'].gt(0)].copy()

        if 'InvoiceNo' in retail:
            retail['IsCancellation'] = (
                retail['InvoiceNo'].astype('string').str.strip().str.upper()
                .str.startswith('C', na=False)
            )
        else:
            retail['IsCancellation'] = False
        retail['LineRevenue'] = retail['Quantity'] * retail['UnitPrice']
        retail['InvoiceYear'] = retail['InvoiceDate'].dt.year
        retail['InvoiceMonth'] = retail['InvoiceDate'].dt.month
        retail['InvoiceDay'] = retail['InvoiceDate'].dt.day

        if 'Description' in retail:
            retail['Description'] = (
                retail['Description'].astype('string').str.strip()
                .replace('', pd.NA).fillna('Unknown product')
            )
        return_or_cancel = retail['Quantity'].lt(0) | retail['IsCancellation']
        online_retail_returns = retail.loc[return_or_cancel].copy()
        zero_quantity = retail['Quantity'].eq(0) & ~retail['IsCancellation']
        online_retail_cleaned = retail.loc[
            retail['Quantity'].gt(0) & ~retail['IsCancellation']
        ].copy()
        cleaned_datasets['Online Retail.xlsx'] = online_retail_cleaned

        cleaning_records.append({
            'dataset': 'Online Retail (loaded preview)',
            'input_rows': retail_input_rows,
            'exact_duplicates_removed': retail_duplicates,
            'invalid_dates_removed': invalid_retail_dates,
            'invalid_numeric_rows_removed': invalid_retail_numeric,
            'nonpositive_prices_removed': nonpositive_retail_prices,
            'returns_or_cancellations_separated': int(return_or_cancel.sum()),
            'zero_quantity_rows_excluded': int(zero_quantity.sum()),
            'retained_sales_rows': len(online_retail_cleaned),
        })

        print('Online Retail rows retained for positive sales:',
              len(online_retail_cleaned))
        print('Return/cancellation rows retained separately:',
              len(online_retail_returns))
        print('Missing CustomerID rows in deduplicated input (not imputed):',
              missing_customer_ids)
        print('Missing CustomerID rows retained in positive sales:',
              int(online_retail_cleaned['CustomerID'].isna().sum())
              if 'CustomerID' in online_retail_cleaned else 'column unavailable')
        print('Missing/blank descriptions labeled Unknown product:',
              missing_descriptions)
        print('Invoice date features added: InvoiceYear, InvoiceMonth, InvoiceDay')

        total_sales_revenue = float(online_retail_cleaned['LineRevenue'].sum())
        if 'InvoiceNo' in online_retail_cleaned:
            invoice_count = int(online_retail_cleaned['InvoiceNo'].nunique())
            average_order_value = (
                total_sales_revenue / invoice_count if invoice_count else 0.0
            )
        else:
            invoice_count = None
            average_order_value = None
        print('Positive-sales revenue:', round(total_sales_revenue, 2))
        print('Sales invoices:', invoice_count)
        print('Average order value:',
              round(average_order_value, 2)
              if average_order_value is not None else 'InvoiceNo column unavailable')
        print('Return/cancellation signed line revenue:',
              round(float(online_retail_returns['LineRevenue'].sum()), 2))

        if 'Country' in online_retail_cleaned:
            print('Top countries by positive-sales revenue:')
            country_summary = (
                online_retail_cleaned.groupby('Country', dropna=False)
                .agg(rows=('LineRevenue', 'size'),
                     total_revenue=('LineRevenue', 'sum'))
                .sort_values('total_revenue', ascending=False)
            )
            display(country_summary.head(10))
        product_column = (
            'Description' if 'Description' in online_retail_cleaned
            else 'StockCode' if 'StockCode' in online_retail_cleaned
            else None
        )
        if product_column:
            print(f'Top products by positive-sales revenue ({product_column}):')
            product_summary = (
                online_retail_cleaned.groupby(product_column, dropna=False)
                .agg(lines=('LineRevenue', 'size'),
                     quantity_sold=('Quantity', 'sum'),
                     total_revenue=('LineRevenue', 'sum'))
                .sort_values('total_revenue', ascending=False)
            )
            display(product_summary.head(10))

if cleaning_records:
    cleaning_summary = pd.DataFrame(cleaning_records).set_index('dataset')
    print('Cleaning counts (loaded rows only; removal stages are sequential):')
    display(cleaning_summary)


## Findings to record

- The large Store Sales CSV is analyzed using the configured 100,000-row preview; `cleaned_datasets["train.csv"]` contains its cleaned copy.
- Online Retail revenue is calculated as `Quantity * UnitPrice` for positive, non-cancelled sales. Returns and cancellation invoices are available separately in `online_retail_returns`.
- `online_retail_cleaned` includes invoice year/month/day features. Unknown customer IDs are retained, and missing descriptions are labeled `Unknown product`.
- Review `cleaning_summary`, revenue, order value, country/product, family/store, and promotion summaries before interpreting results.
